# INSTALL / IMPORT LIBRARIES

In [0]:
%pip install simple-salesforce

In [0]:
import json
import os

config_path ='/Workspace/Users/kortum.facturas@gmail.com/Drafts/GAMBILL PROJECTS/3RD PROJECT/config.json'
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = json.load(f)

shopify_name = config.get("SHOPIFY_STORE")
shopify_name_ver = '2026-04'
shopify_token = config.get("X-Shopify-Access-Token")

salesforce_name = config.get("SALESFORCE")
salesforce_key = config.get("SALESFORCE_API_KEY")
salesforce_secret_key = config.get("SALESFORCE_API_KEY_SECRET")
salesforce_user = config.get("SALESFORCE_USER")
salesforce_pass = config.get("SALESFORCE_PASS")
salesforce_tok = config.get("SALESFORCE_TOKEN")

In [0]:
import requests
import datetime
import time
import logging
import copy
from pyspark.sql import Window
from pyspark.sql import Row
from delta.tables import DeltaTable
from pyspark.sql import functions as F

start_time = time.perf_counter()


# BRONZE LAYER / API REQUEST FROM THE 3 DIFFERENT PLATFORMS

In [0]:

zendesk_volume_path ='/Volumes/bronze/customer_master_data_management/zendesk_customers'
df_zendesk = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(zendesk_volume_path)
df_zendesk = df_zendesk.withColumn("zd_user_id", F.col("zd_user_id").cast("string"))
df_zendesk.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('bronze.customer_master_data_management.zendesk_customers_table')
# display(df_zendesk)

In [0]:
url = f"https://{shopify_name}.myshopify.com/admin/api/{shopify_name_ver}/customers.json"
headers = {
    "X-Shopify-Access-Token": shopify_token,
    "Content-Type": "application/json"
}

LIMIT = 250
customers_data = []
# params = {"limit": LIMIT}
params = {"limit": LIMIT,
          "fields": "id, first_name, last_name, phone, email, default_address"} # Define the needed fields / makes it faster to read
has_next_page = True

while has_next_page:
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        customers = data.get("customers", [])
        customers_data.extend(customers)
        # print(customers_data)
        # print(response.headers)
        # Capture the Link header
        link_header = response.headers.get("Link")
        next_url = None
        
        # Check if a link header exists
        if link_header:
            # Shopify headers look like: <URL>; rel="previous", <URL>; rel="next"
            # Split by commas to evaluate links individually
            links = link_header.split(",")
            for link in links:
                if 'rel="next"' in link:
                    # Isolate just the URL inside the angle brackets <>
                    next_url = link.split(";")[0].strip("<> ")
        
        # Control loop continuation cleanly
        if next_url:
            url = next_url
            params = {}  # Keep empty; next_url has built-in page_info
        else:
            has_next_page = False  # Safely breaks loop when no next page exists
            
    else:
        print(f"Error fetching data: {response.status_code}")
        break

# Explicit Schema declaration
from pyspark.sql.types import StructType, StructField, LongType, StringType
shopify_schema = StructType([
    StructField("id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("default_address", StructType([
        StructField("city", StringType(), True),
        StructField("province_code", StringType(), True),
        StructField("zip", StringType(), True),
        StructField("phone", StringType(), True),
        StructField("address1", StringType(), True)
    ]), True)
])

# Convert to Dataframe
df_raw = spark.createDataFrame(customers_data, schema=shopify_schema)

# 3. Clean and flatten the columns
df_flat = df_raw.select(
    F.col("id").alias("customer_id"),
    F.col("first_name"),
    F.col("last_name"),
    F.col("email"),
    F.col("default_address.city").alias("city"),
    F.col("default_address.province_code").alias("state"),
    F.col("default_address.zip").alias("zip"),
    F.col("default_address.phone").alias("phone"),
    F.col("default_address.address1").alias("address")
)

df_flat.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('bronze.customer_master_data_management.shopify_customers_table')
# display(df_flat)

In [0]:

# [{'id': 10369863942328, 
#   'created_at': '2026-06-19T19:11:53-04:00', 
#   'updated_at': '2026-06-19T19:11:53-04:00', 
#   'first_name': 'Jason', 
#   'last_name': 'Andrade', 
#   'orders_count': 0, 
#   'state': 'disabled', 
#   'total_spent': '0.00', 
#   'last_order_id': None, 
#   'note': None, 
#   'verified_email': True, 
#   'multipass_identifier': None, 
#   'tax_exempt': False, 
#   'tags': '', 
#   'last_order_name': None, 
#   'email': 'jason.andrade@gmail.com', 
#   'phone': None, 
#   'currency': 'USD', 
#   'addresses': [{'id': 12235531092152, 'customer_id': 10369863942328, 'first_name': 'Jason', 'last_name': 'Andrade', 
#                  'company': None,   'address1': None, 'address2': None, 'city': 'Burketown', 'province': None, 'country': None, 'zip': '35552', 'phone': None, 'name': 'Jason Andrade', 'province_code': None, 'country_code': None, 'country_name': None, 'default': True}], 
#   'tax_exemptions': [], 
#   'email_marketing_consent': {'state': 'not_subscribed', 'opt_in_level': 'single_opt_in', 'consent_updated_at': None}, 'sms_marketing_consent': None, 
#   'admin_graphql_api_id': 'gid://shopify/Customer/10369863942328', 
#   'default_address': {'id': 12235531092152, 'customer_id': 10369863942328, 'first_name': 'Jason', 'last_name': 'Andrade', 
#                       'company': None, 'address1': None, 'address2': None, 'city': 'Burketown', 'province': None, 'country': None, 'zip': '35552', 'phone': None, 'name': 'Jason Andrade', 'province_code': None, 'country_code': None, 'country_name': None, 'default': True}}]

In [0]:
from simple_salesforce import Salesforce
from pyspark.sql.types import StructType, StructField, StringType

# Define credentials 
USER = salesforce_user
PWD = salesforce_pass
TOK = salesforce_tok  
KEY = salesforce_key
SEC = salesforce_secret_key

print("Initializing Secure Connection to Salesforce Production/Developer Instance...")

try:
    sf = Salesforce(
        username=USER,
        password=PWD,
        security_token=TOK,
        consumer_key=KEY,
        consumer_secret=SEC,
        domain="login"
    )
    print("SUCCESS: Successfully authenticated to Production!")

    print("Extracting ALL unconverted lead data via API query...")
    
    # 'query_all' to automatically paginate beyond the 2,000 row limit
    leads = sf.query_all("SELECT sfdc_id__c, FirstName, LastName, City, State, Phone, Email FROM Lead ")
    
    if leads['records']:
        # Format rows cleanly, dropping Salesforce system metadata attributes
        cleaned_records = [{k: v for k, v in row.items() if k != 'attributes'} for row in leads['records']]
        
        # Explicit Schema declaration
        schema = StructType([
            StructField("sfdc_id__c", StringType(), True),
            StructField("FirstName", StringType(), True),
            StructField("LastName", StringType(), True),
            StructField("City", StringType(), True),
            StructField("State", StringType(), True),
            StructField("Phone", StringType(), True),
            StructField("Email", StringType(), True)
        ])
        
        spark_df = spark.createDataFrame(cleaned_records, schema=schema)
        
        # Verify the record count in your console output
        print(f"Successfully processed {spark_df.count()} leads into Databricks Spark Engine.")

        # display(spark_df)
    else:
        print("API Connection successful, but zero unconverted leads matched your criteria.")

except Exception as e:
    raise Exception(f"Production Pipeline Pipeline Terminated: {str(e)}")

spark_df.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('bronze.customer_master_data_management.salesforce_leads_table')

# SILVER LAYER / STANDARDIZATION

In [0]:
#---READ THE BRONZE TABLES INTO DATAFRAMES
silver_zendesk = spark.read.table('bronze.customer_master_data_management.zendesk_customers_table')
silver_shopify = spark.read.table('bronze.customer_master_data_management.shopify_customers_table')
silver_salesforce = spark.read.table('bronze.customer_master_data_management.salesforce_leads_table')


#---APPLY PREPROCESSING AND STANDRADIZATION
# CONVERT STATE NAMES TO STATE CODES
state_code_state_name = spark.read.csv('/Volumes/bronze/customer_master_data_management/state_name_vs_state_code', header=True, inferSchema=True)
silver_salesforce = silver_salesforce.join(state_code_state_name, on='State', how='left').drop('state')

# CONVERT COLUMN NAMES TO LOWERCASE AND CHANGE COLUMNS NAME TO MATCH THE OTHER TABLES
silver_salesforce = silver_salesforce.toDF(*[col.lower() for col in silver_salesforce.columns])
silver_salesforce = silver_salesforce.withColumnsRenamed({'firstname': 'first_name', 'lastname': 'last_name', 'abbreviation': 'state'})
# display(silver_salesforce)

# CONVERT ALL TEXT TO LOWERCASE AND REMOVE SPACES
target_columns = ['first_name', 'last_name', 'email']
for col in target_columns:
    silver_salesforce = silver_salesforce.withColumn(col, F.lower(F.col(col)))
    silver_salesforce = silver_salesforce.withColumn(col, F.trim(F.col(col)))
    silver_salesforce = silver_salesforce.withColumn(col, F.regexp_replace(F.col(col), r'\s+', ''))

    silver_zendesk = silver_zendesk.withColumn(col, F.lower(F.col(col)))
    silver_zendesk = silver_zendesk.withColumn(col, F.trim(F.col(col)))
    silver_zendesk = silver_zendesk.withColumn(col, F.regexp_replace(F.col(col), r'\s+', ''))

    silver_shopify = silver_shopify.withColumn(col, F.lower(F.col(col)))
    silver_shopify = silver_shopify.withColumn(col, F.trim(F.col(col)))
    silver_shopify = silver_shopify.withColumn(col, F.regexp_replace(F.col(col), r'\s+', ''))

# CONVERT PHONE NUMBERS TO E.164 FORMAT
silver_salesforce = silver_salesforce.withColumn('phone', F.regexp_replace(F.col('phone'), r'[ x+()-.]', ''))
silver_shopify = silver_shopify.withColumn('phone', F.regexp_replace(F.col('phone'), r'[ x+()-.]', ''))
# FIRST MAKE SURE ALL NUMBERS ARE 10 DIGITS LONG AND REMOVE THE EXTENSION. THEN I WILL ADD THE '+1'
silver_salesforce = silver_salesforce.withColumn('phone', F.regexp_replace(F.col('phone'), r'^001', ''))
silver_salesforce = silver_salesforce.withColumn('phone', F.regexp_replace(F.col('phone'), r'^[01]', ''))

silver_shopify = silver_shopify.withColumn('phone', F.regexp_replace(F.col('phone'), r'^001', ''))
silver_shopify = silver_shopify.withColumn('phone', F.regexp_replace(F.col('phone'), r'^[01]', ''))

# SEPARATE EXTENSION FROM PHONE NUMBER (MAYBE I SHOULD IGNORE THE EXTENSION)
silver_salesforce = silver_salesforce.withColumn('phone_number', F.substring(F.col('phone'),1, 10))
silver_salesforce = silver_salesforce.withColumn('phone_number', F.concat(F.lit('+1'), F.col('phone_number')))
silver_salesforce = silver_salesforce.withColumn('phone_ext', F.substring(F.col('phone'),11, 100)).drop('phone')

silver_shopify = silver_shopify.withColumn('phone_number', F.substring(F.col('phone'),1, 10))
silver_shopify = silver_shopify.withColumn('phone_number', F.concat(F.lit('+1'), F.col('phone_number')))
silver_shopify = silver_shopify.withColumn('phone_ext', F.substring(F.col('phone'),11, 100)).drop('phone')

# IF STATE IS EMPTY WE CAN USE THE PHONE NUMBER TO ASUME THE STATE, THE ADDRES, ZIP CODE

display(silver_salesforce[['sfdc_id__c']].distinct().count())
display(silver_salesforce)
display(silver_shopify[['customer_id']].distinct().count())
display(silver_shopify)
display(silver_zendesk[['zd_user_id']].distinct().count())
display(silver_zendesk)

#                                                           DETERMINISTIC MATCHING BLOCK

In [0]:
# SINCE WE ARE USING THE EMAIL FIELD AS OUR DETERMINISTIC BLOCK FIRST I NEED TO FIND IF THERE ARE ANY DUPLICATES
# IT WILL AVOID CREATING MULTIPLE LINES IN MY MERGE STATEMENT

shopify_dup_emails = silver_shopify.groupby('email').count().filter('count > 1').drop('count')
silver_shopify_no_dup_emails = silver_shopify[~silver_shopify['email'].isin(shopify_dup_emails)]
silver_shopify_dup_emails = silver_shopify[silver_shopify['email'].isin(shopify_dup_emails)]


salesforce_dup_emails = silver_salesforce.groupby('email').count().filter('count > 1').drop('count')
silver_salesforce_no_dup_emails = silver_salesforce[~silver_salesforce['email'].isin(salesforce_dup_emails)]
silver_salesforce_dup_emails = silver_salesforce[silver_salesforce['email'].isin(salesforce_dup_emails)]


zendesk_dup_emails = silver_zendesk.groupby('email').count().filter('count > 1').drop('count')
silver_zendesk_no_dup_emails = silver_zendesk[~silver_zendesk['email'].isin(zendesk_dup_emails)]
silver_zendesk_dup_emails = silver_zendesk[silver_zendesk['email'].isin(zendesk_dup_emails)]

display(silver_zendesk_dup_emails)


# MATERIALIZE deterministic match results to temp tables to avoid recomputing
# Write to temp tables 
silver_shopify_no_dup_emails.write.mode('overwrite').saveAsTable('silver.customer_master_data_management.temp_silver_shopify_no_dup_emails')
silver_salesforce_no_dup_emails.write.mode('overwrite').saveAsTable('silver.customer_master_data_management.temp_silver_salesforce_no_dup_emails')
silver_zendesk_no_dup_emails.write.mode('overwrite').saveAsTable('silver.customer_master_data_management.temp_silver_zendesk_no_dup_emails')

# Read back from temp tables
silver_shopify_no_dup_emails = spark.table('silver.customer_master_data_management.temp_silver_shopify_no_dup_emails')
silver_salesforce_no_dup_emails = spark.table('silver.customer_master_data_management.temp_silver_salesforce_no_dup_emails')
silver_zendesk_no_dup_emails = spark.table('silver.customer_master_data_management.temp_silver_zendesk_no_dup_emails')


df_primary = silver_shopify_no_dup_emails.alias('primary')
df_secondary = silver_salesforce_no_dup_emails.alias('secondary')
df_third = silver_zendesk_no_dup_emails.alias('third')


In [0]:
# JOIN SHOPIFY / SALESFORCE / ZENDESK DETERMINISTIC MATCHES
exact_shop_sf_zd = silver_shopify_no_dup_emails.join(silver_salesforce_no_dup_emails, on="email", how="inner") \
                                            .join(silver_zendesk_no_dup_emails, on="email", how="inner") \
    .select("customer_id", "sfdc_id__c", "zd_user_id", F.lit(1.0).alias("confidence_score"), F.lit("deterministic_email").alias("match_method"))

exact_shop_sf_zd_1 = exact_shop_sf_zd.join(df_primary, on='customer_id', how='left').alias('first')\
    .join(df_secondary, on='sfdc_id__c', how='left')\
    .join(df_third, on='zd_user_id', how='left')\
    .select(
        'first.customer_id', 
        'first.sfdc_id__c', 
        'first.zd_user_id', 
        'first.confidence_score', 
        'first.match_method',
        F.coalesce('first.first_name', 'secondary.first_name', 'third.first_name', F.lit("Unknown")).alias('first_name'), 
        F.coalesce('first.last_name', 'secondary.last_name', 'third.last_name', F.lit("Unknown")).alias('last_name'), 
        F.coalesce('first.email', 'secondary.email', 'third.email', F.lit("Unknown")).alias('email'),
        F.coalesce('first.city', 'secondary.city', F.lit("Unknown")).alias('city'),
        F.coalesce('first.state', 'secondary.state', F.lit("Unknown")).alias('state'),
        F.coalesce('first.zip', F.lit("Unknown")).alias('zip'),
        F.coalesce('first.address',F.lit('Unknown')).alias('address'),
        F.coalesce('first.phone_number', 'secondary.phone_number', F.lit("Unknown")).alias('phone_number'),
        F.coalesce('first.phone_ext', 'secondary.phone_ext', F.lit("Unknown")).alias('phone_ext')
        )


mask = ~silver_shopify_no_dup_emails['customer_id'].isin(exact_shop_sf_zd[['customer_id']])

exact_shop_sf = silver_shopify_no_dup_emails[mask].join(silver_salesforce_no_dup_emails, on="email", how="inner") \
    .select("customer_id", "sfdc_id__c", F.lit('N/A').alias("zd_user_id"), F.lit(1.0).alias("confidence_score"), F.lit("deterministic_email").alias("match_method"))

exact_shop_sf2 = exact_shop_sf.join(df_primary, on='customer_id', how='left').alias('first')\
    .join(df_secondary, on='sfdc_id__c', how='left')\
    .select(
        'first.customer_id', 
        'first.sfdc_id__c', 
        'first.zd_user_id', 
        'first.confidence_score', 
        'first.match_method',
        F.coalesce('first.first_name', 'secondary.first_name', F.lit("Unknown")).alias('first_name'), 
        F.coalesce('first.last_name', 'secondary.last_name', F.lit("Unknown")).alias('last_name'), 
        F.coalesce('first.email', 'secondary.email', F.lit("Unknown")).alias('email'),
        F.coalesce('first.city', 'secondary.city', F.lit("Unknown")).alias('city'),
        F.coalesce('first.state', 'secondary.state', F.lit("Unknown")).alias('state'),
        F.coalesce('first.zip', F.lit("Unknown")).alias('zip'),
        F.coalesce('first.address',F.lit('Unknown')).alias('address'),
        F.coalesce('first.phone_number', 'secondary.phone_number', F.lit("Unknown")).alias('phone_number'),
        F.coalesce('first.phone_ext', 'secondary.phone_ext', F.lit("Unknown")).alias('phone_ext')
        )


exact_shop_zd = silver_shopify_no_dup_emails[mask].join(silver_zendesk_no_dup_emails, on="email", how="inner") \
    .select("customer_id", F.lit('N/A').alias("sfdc_id__c"), "zd_user_id", F.lit(1.0).alias("confidence_score"), F.lit("deterministic_email").alias("match_method"))

exact_shop_zd2 = exact_shop_zd.join(df_primary, on='customer_id', how='left').alias('first')\
    .join(df_third, on='zd_user_id', how='left')\
    .select(
        'first.customer_id', 
        'first.sfdc_id__c', 
        'first.zd_user_id', 
        'first.confidence_score', 
        'first.match_method',
        F.coalesce('first.first_name', 'third.first_name', F.lit("Unknown")).alias('first_name'), 
        F.coalesce('first.last_name', 'third.last_name', F.lit("Unknown")).alias('last_name'), 
        F.coalesce('first.email', 'third.email', F.lit("Unknown")).alias('email'),
        F.coalesce('first.city', F.lit("Unknown")).alias('city'),
        F.coalesce('first.state', F.lit("Unknown")).alias('state'),
        F.coalesce('first.zip', F.lit("Unknown")).alias('zip'),
        F.coalesce('first.address',F.lit('Unknown')).alias('address'),
        F.coalesce('first.phone_number', F.lit("Unknown")).alias('phone_number'),
        F.coalesce('first.phone_ext', F.lit("Unknown")).alias('phone_ext')
        )

# First part of the golden records in the silver layer
silver_deterministic_golden_recods = exact_shop_sf_zd_1.unionByName(exact_shop_sf2).unionByName(exact_shop_zd2)
display(silver_deterministic_golden_recods)

# THIS ARE ALL THE UNMATCHED RECORDS
matched_shop_ids = silver_deterministic_golden_recods.select("customer_id")
unmatched_shop = silver_shopify_no_dup_emails.join(matched_shop_ids, on="customer_id", how="left_anti")

matched_sf_ids = silver_deterministic_golden_recods.select("sfdc_id__c")
unmatched_sf = silver_salesforce_no_dup_emails.join(matched_sf_ids, on="sfdc_id__c", how="left_anti")
unmatched_sf = unmatched_sf.select([F.col(c).alias(f"sf_{c}" if c != 'sfdc_id__c' else c) for c in unmatched_sf.columns])

matched_zd_ids = silver_deterministic_golden_recods.select("zd_user_id")
unmatched_zd = silver_zendesk_no_dup_emails.join(matched_zd_ids, on="zd_user_id", how="left_anti")
unmatched_zd = unmatched_zd.select([F.col(c).alias(f"zd_{c}" if c != 'zd_user_id' else c) for c in unmatched_zd.columns])

display(matched_shop_ids.select("customer_id").filter(F.col('customer_id')!='N/A'))
display(matched_sf_ids.select("sfdc_id__c").filter(F.col('sfdc_id__c')!='N/A'))
display(matched_zd_ids.select("zd_user_id").filter(F.col('zd_user_id')!='N/A'))



# MATERIALIZE deterministic match results to temp tables to avoid recomputing
# Write to temp tables 
unmatched_shop.write.mode('overwrite').saveAsTable('silver.customer_master_data_management.temp_unmatched_shop')
unmatched_sf.write.mode('overwrite').saveAsTable('silver.customer_master_data_management.temp_unmatched_sf')
unmatched_zd.write.mode('overwrite').saveAsTable('silver.customer_master_data_management.temp_unmatched_zd')

# Read back from temp tables
unmatched_shop = spark.table('silver.customer_master_data_management.temp_unmatched_shop')
unmatched_sf = spark.table('silver.customer_master_data_management.temp_unmatched_sf')
unmatched_zd = spark.table('silver.customer_master_data_management.temp_unmatched_zd')


display(unmatched_shop)
display(unmatched_sf)
display(unmatched_zd)

In [0]:
# ZIP AND ADDRESS IS SOMETHING THAT CANT BE FILLED USING HIREARCHY BECAUSE THERE IS NO OTHER SOURCE
# filtered = silver_shopify_no_dup_emails[silver_shopify_no_dup_emails['customer_id'].isin(exact_shop_sf_zd2[['customer_id']])]
# # display(filtered)
# # display(filtered.filter(F.col('address').isNull()))

# null_counts = filtered.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in filtered.columns])
# display(null_counts)

# null_counts2 = exact_shop_sf_zd2.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in exact_shop_sf_zd2.columns])
# display(null_counts2)


# FUZZY MATCHING BLOCK

In [0]:
fuzz_matching_types_define = {'soundex_score': {'score_value'   : 20,
                                         'match_column_names'   : ['first_name', 'sf_first_name', 'last_name','sf_last_name']
                                        }, 
                       'leven_name_score': {'score_value'       : 15,
                                         'match_column_names'   : ['first_name', 'sf_first_name']
                                        },
                       'leven_last_name_score': {'score_value'  : 15,
                                         'match_column_names'   : ['last_name','sf_last_name']
                                        },
                        'email_score':   {'score_value'         : 30,
                                         'match_column_names'   : ['email', 'sf_email']
                                        },
                       'phone_score':   {'score_value'          : 10,
                                         'match_column_names'   : ['phone_number', 'sf_phone_number']
                                        },
                       'leven_city_score': {'score_value'       : 5,
                                         'match_column_names'   : ['city', 'sf_city']
                                        },
                       'state_score':   {'score_value'          : 5,
                                         'match_column_names'   : ['state', 'sf_state']
                                        },
                        }

def define_the_fuzz_matching_types(df1, df2) -> dict:
    # A list of specific prefixes to look for
    prefixes_list = ["sf_", "zd_"]

    rename_map = {}
    df1_prefix =''
    for col in df1.columns:
        for prefix in prefixes_list:
            if col.startswith(prefix):
                # Map the original name to the clean version
                rename_map[col] = col.removeprefix(prefix)
                df1_prefix = prefix
                break  # Stop checking other prefixes once a match is found

    # Rename all columns in a single step
    df1_cleaned = df1.withColumnsRenamed(rename_map)

    rename_map = {}
    df2_prefix =''
    for col in df2.columns:
        for prefix in prefixes_list:
            if col.startswith(prefix):
                # Map the original name to the clean version
                rename_map[col] = col.removeprefix(prefix)
                df2_prefix = prefix
                break  # Stop checking other prefixes once a match is found

    # Rename all columns in a single step
    df2_cleaned = df2.withColumnsRenamed(rename_map)


    matching_columns = list(set(df1_cleaned.columns).intersection(df2_cleaned.columns))
    # print(f"Found {len(matching_columns)} columns with the same name:")
    # print(matching_columns)

    matching_types_available={}
    matching_types_list = []
    total_points = 0
    
    if 'first_name' in matching_columns and 'last_name' in matching_columns:
        matching_types_available['soundex_score'] = (fuzz_matching_types_define['soundex_score']).copy()
        matching_types_available['soundex_score']['match_column_names'] = [df1_prefix + 'first_name', df2_prefix + 'first_name', df1_prefix + 'last_name', df2_prefix + 'last_name']

        matching_types_available['leven_name_score'] = (fuzz_matching_types_define['leven_name_score']).copy()
        matching_types_available['leven_name_score']['match_column_names'] = [df1_prefix + 'first_name', df2_prefix + 'first_name']

        matching_types_available['leven_last_name_score'] = (fuzz_matching_types_define['leven_last_name_score']).copy()
        matching_types_available['leven_last_name_score']['match_column_names'] = [df1_prefix + 'last_name', df2_prefix + 'last_name']

        matching_types_list.extend(['soundex_score', 'leven_name_score', 'leven_last_name_score'])
        total_points = total_points + matching_types_available['soundex_score']['score_value'] + matching_types_available['leven_name_score']['score_value'] + matching_types_available['leven_last_name_score']['score_value']

    if 'email' in matching_columns :
        matching_types_available['email_score'] = (fuzz_matching_types_define['email_score']).copy()
        matching_types_available['email_score']['match_column_names'] = [df1_prefix + 'email', df2_prefix + 'email']

        matching_types_list.extend(['email_score'])
        total_points = total_points + matching_types_available['email_score']['score_value']

    if 'city' in matching_columns :
        matching_types_available['leven_city_score'] = (fuzz_matching_types_define['leven_city_score']).copy()
        matching_types_available['leven_city_score']['match_column_names'] = [df1_prefix + 'city', df2_prefix + 'city']

        matching_types_list.extend(['leven_city_score'])
        total_points = total_points + matching_types_available['leven_city_score']['score_value']

    if 'phone_number' in matching_columns :
        matching_types_available['phone_score'] = (fuzz_matching_types_define['phone_score']).copy()
        matching_types_available['phone_score']['match_column_names'] = [df1_prefix + 'phone_number', df2_prefix + 'phone_number']

        matching_types_list.extend(['phone_score'])
        total_points = total_points + matching_types_available['phone_score']['score_value']

    if 'state' in matching_columns :
        matching_types_available['state_score'] = (fuzz_matching_types_define['state_score']).copy()
        matching_types_available['state_score']['match_column_names'] = [df1_prefix + 'state', df2_prefix + 'state']

        matching_types_list.extend(['state_score'])
        total_points = total_points + matching_types_available['state_score']['score_value']

    # print(total_points)
    # Need to normalize weight because not all columns are being selected
    for types in matching_types_list:
        matching_types_available[types]['score_value'] = matching_types_available[types]['score_value']*100/total_points


    return matching_types_available



#---------------------LEARN ABOUT THE JARO-WINKLER
# 3. Vectorized Fuzzy Algorithm Logic

def get_scoring_expression(score_type: str, column_names: list, score_value: int) -> F.Column:
    """
    Returns a native PySpark Column expression based on the requested score type.
    """
    # Use if/elif blocks to prevent eager evaluation of out-of-bounds list indices
    # Soundex Component (Strict Binary match)
    if score_type == "soundex_score":
        return F.when(
            (F.soundex(column_names[0]) == F.soundex(column_names[1])) & 
            (F.soundex(column_names[2]) == F.soundex(column_names[3])), 
            score_value
        ).otherwise(0)
    # Normalized Levenshtein Name Distance component 
    elif score_type == "leven_name_score":
        return F.round(
            (1.0 - (F.levenshtein(column_names[0], column_names[1]) / 
            F.greatest(F.greatest(F.length(column_names[0]), F.length(column_names[1])), F.lit(1)))) * score_value
        )
    # Normalized Levenshtein Name Distance component
    elif score_type == "leven_last_name_score":
        return F.round(
            (1.0 - (F.levenshtein(column_names[0], column_names[1]) / 
            F.greatest(F.greatest(F.length(column_names[0]), F.length(column_names[1])), F.lit(1)))) * score_value
        )
    elif score_type == "email_score":
        return F.round(
            (1.0 - (F.levenshtein(column_names[0], column_names[1]) / 
            F.greatest(F.greatest(F.length(column_names[0]), F.length(column_names[1])), F.lit(1)))) * score_value
        )
    # Exact Phone matching value multiplier component  
    elif score_type == "phone_score":
        return F.when(
            F.col(column_names[0]).isNotNull() & (F.col(column_names[0]) == F.col(column_names[1])), 
            score_value
        ).otherwise(0)
        
    elif score_type == "leven_city_score":
        return F.round(
            (1.0 - (F.levenshtein(column_names[0], column_names[1]) / 
            F.greatest(F.greatest(F.length(column_names[0]), F.length(column_names[1])), F.lit(1)))) * score_value
        )
    # Exact state matching    
    elif score_type == "state_score":
        return F.when(
            F.col(column_names[0]).isNotNull() & (F.col(column_names[0]) == F.col(column_names[1])), 
            score_value
        ).otherwise(0)
        
    else:
        raise ValueError(f"Unknown score type: '{score_type}'.")


In [0]:
# BLOCK AND INDEX RECORDS (MAYBE USE THE FIRST NAME LETTER AND THE LAST NAME)

def test_for_different_combination(df1, df2, confidence_threshold = [0.85, 0.75]):

    fuzz_matching_types = define_the_fuzz_matching_types(df1, df2)

# for i in range(3):
    
#     if i == 0:
#         # # THIS GAVE ME 100K rows need to be computed
#         unmatched_shop_blocked = df1.withColumn("block_key", F.substring(F.col(fuzz_matching_types['leven_last_name_score']['match_column_names'][0]), 1, 1))
#         unmatched_sf_blocked = df2.withColumn("block_key", F.substring(F.col(fuzz_matching_types['leven_last_name_score']['match_column_names'][1]), 1, 1))
#     elif i == 1:
#         # # THIS GAVE ME 25K rows need to be computed
#         unmatched_shop_blocked = df1.withColumn("block_key", F.substring(F.col(fuzz_matching_types['leven_last_name_score']['match_column_names'][0]), 1, 2))
#         unmatched_sf_blocked = df2.withColumn("block_key", F.substring(F.col(fuzz_matching_types['leven_last_name_score']['match_column_names'][1]), 1, 2))
#     else:
#         # THIS GAVE ME 8K rows need to be computed
#         unmatched_shop_blocked = df1.withColumn("block_key", F.concat(F.substring(F.col(fuzz_matching_types['leven_last_name_score']['match_column_names'][0]), 1, 1), 
#                                                                                 F.substring(F.col(fuzz_matching_types['leven_name_score']['match_column_names'][0]), 1, 1)))
#         unmatched_sf_blocked = df2.withColumn("block_key", F.concat(F.substring(F.col(fuzz_matching_types['leven_last_name_score']['match_column_names'][1]), 1, 1), 
#                                                                             F.substring(F.col(fuzz_matching_types['leven_name_score']['match_column_names'][1]), 1, 1)))

    # THIS GAVE ME 8K rows need to be computed
    if 'leven_last_name_score' in fuzz_matching_types or 'leven_name_score' in fuzz_matching_types:
        unmatched_shop_blocked = df1.withColumn("block_key", F.concat(F.substring(F.col(fuzz_matching_types['leven_last_name_score']['match_column_names'][0]), 1, 1), 
                                                                                F.substring(F.col(fuzz_matching_types['leven_name_score']['match_column_names'][0]), 1, 1)))
        unmatched_sf_blocked = df2.withColumn("block_key", F.concat(F.substring(F.col(fuzz_matching_types['leven_last_name_score']['match_column_names'][1]), 1, 1), 
                                                                            F.substring(F.col(fuzz_matching_types['leven_name_score']['match_column_names'][1]), 1, 1)))
    else:
        unmatched_shop_blocked = df1.withColumn("block_key", F.substring(F.col(fuzz_matching_types['email_score']['match_column_names'][0]), 1, 1))
        unmatched_sf_blocked = df2.withColumn("block_key", F.substring(F.col(fuzz_matching_types['email_score']['match_column_names'][1]), 1, 1))
    
    # JOIN BASED ON THE BLOCK KEY
    candidate_pairs = unmatched_shop_blocked.join(
        unmatched_sf_blocked, 
        on="block_key",
        how='full', 
    )
    # display(candidate_pairs)

    fuzzy_scored_pairs = candidate_pairs
    fuzzy_scored_pairs = fuzzy_scored_pairs.withColumn('total_confidence', F.lit(0))
    for keys in fuzz_matching_types:
        fuzzy_scored_pairs = fuzzy_scored_pairs.withColumn(keys, get_scoring_expression(keys, 
                                                                                    fuzz_matching_types[keys]['match_column_names'],
                                                                                    fuzz_matching_types[keys]['score_value']))
        fuzzy_scored_pairs = fuzzy_scored_pairs.withColumn('total_confidence', F.col(keys)/100 + F.col('total_confidence'))
    
    # display(fuzzy_scored_pairs)
    # target_cols = ['customer_id', 'sf_sfdc_id__c', 'zd_user_id']
    # result = fuzzy_scored_pairs[fuzzy_scored_pairs['total_confidence']>=0.85]

    # 1. Identify which customer ID column exists in the DataFrame
    if "customer_id" in fuzzy_scored_pairs.columns:
        partition_col = "customer_id"
    elif "sfdc_id__c" in fuzzy_scored_pairs.columns:
        partition_col = "sfdc_id__c"
    else:
        raise ValueError("Neither 'customer_id' nor 'sfdc_id__c' found in DataFrame columns!")

    # Use the window function to rank the customers id based on the total confidence
    windowSpec = Window.partitionBy(partition_col).orderBy(F.col("total_confidence").desc())
    # # Apply the rank function (only get the highest rank among the same customer id) sometimes there will be a couple with the same rank
    # fuzzy_scored_pairs2 = fuzzy_scored_pairs.withColumn("rank", F.rank().over(windowSpec)).filter(F.col("rank") == 1)
    # Apply row_number (NOT rank) to guarantee exactly ONE match per customer_id, even with ties
    fuzzy_scored_pairs2 = fuzzy_scored_pairs.withColumn("rank", F.row_number().over(windowSpec)).filter(F.col("rank") == 1)

    # bins = [-1, 74, 84, 100]
    # labels = ['74% and Below', '75% - 84%', '85% and Above']

    thresholds = sorted(confidence_threshold)
    
    # 2. Extract lower and upper cutoffs
    lower_pct = thresholds[0]
    upper_pct = thresholds[1]
    
    # 3. Dynamically build the labels list
    labels = [
        f'{(lower_pct - 0.01)*100}% and Below',
        f'{lower_pct*100}% - {(upper_pct - 0.01)*100}%',
        f'{upper_pct*100}% and Above'
    ]

    df_with_ranges = fuzzy_scored_pairs2.withColumn(
        'confidence_range',
        F.when(F.col('total_confidence') >= upper_pct, labels[2])
        .when((F.col('total_confidence') >= lower_pct) & (F.col('total_confidence') < upper_pct), labels[1])
        .otherwise(labels[0])
    )

    summary_table = df_with_ranges.groupby('confidence_range')\
                                    .count()\
                                    .withColumnRenamed('count','row_count')
    # display(summary_table) # The row count will show more than th actual amount of customers id because of the full join and there are will be nan values

    return fuzzy_scored_pairs2.withColumn("match_method", F.lit("fuzzy")).drop('block_key')

   


In [0]:
### JUST DO THE FUZZY MATCHING BASED ON THE EMAIL COLUMN / LETS SET THE CONFIDENSE HIGH TO 95% MAKE SURE THERE IS ONLY A ONE LETTER MISTAKE
unmatched_shop_email = unmatched_shop.select(['customer_id', 'email'])
unmatched_sf_email = unmatched_sf.select(['sfdc_id__c', 'sf_email'])
unmatched_zd_email = unmatched_zd.select(['zd_user_id', 'zd_email'])

# display(unmatched_shop_email)
# display(unmatched_sf_email)
# display(unmatched_zd_email)

fuzzy_match_email_shop_sf = test_for_different_combination(unmatched_shop_email, unmatched_sf_email, [0.95, 0.85])
fuzzy_match_email_shop_zd = test_for_different_combination(unmatched_shop_email, unmatched_zd_email, [0.95, 0.85])
fuzzy_match_email_sf_zd = test_for_different_combination(unmatched_sf_email, unmatched_zd_email, [0.95, 0.85])


fuzzy_match_email_shop_sf2 = fuzzy_match_email_shop_sf.filter(F.col('total_confidence')>=0.95).select(['customer_id', 'sfdc_id__c'])
fuzzy_match_email_shop_zd2 = fuzzy_match_email_shop_zd.filter(F.col('total_confidence')>=0.95).select(['customer_id', 'zd_user_id'])
fuzzy_match_email_sf_zd2 = fuzzy_match_email_sf_zd.filter(F.col('total_confidence')>=0.95).select(['sfdc_id__c', 'zd_user_id'])


# 1. First full join on customer_id
fuzy_match_email_all = fuzzy_match_email_shop_sf2.join(fuzzy_match_email_shop_zd2, ['customer_id'], "full")

# 2. Rename zd_user_id in the incoming DataFrame to avoid namespace collision
fuzzy_match_email_sf_zd2_renamed = fuzzy_match_email_sf_zd2.withColumnRenamed('zd_user_id', 'zd_user_id_sf')

# 3. Second full join on sfdc_id__c
fuzy_match_email_all2 = fuzy_match_email_all.join(fuzzy_match_email_sf_zd2_renamed, ['sfdc_id__c'], "full")

# 4. Merge the duplicate columns into one and drop the temporary renamed column
final_output = fuzy_match_email_all2.withColumn(
    'zd_user_id', 
    F.coalesce(F.col('zd_user_id'), F.col('zd_user_id_sf'))
).drop('zd_user_id_sf')

# Display the clean final output
# display(final_output)




In [0]:
### REMOVE COLUMN FOUND IN THE FUZZY EMAIL MATCHING
# unmatched_shop2 = unmatched_shop[~unmatched_shop['customer_id'].isin(final_output['customer_id'])] # This is not working i had to use anti join
# print(final_output.select('customer_id').dropna().distinct().count())
# print(final_output.select('sfdc_id__c').dropna().distinct().count())
# print(final_output.select('zd_user_id').dropna().distinct().count())
unmatched_shop2 = unmatched_shop.join(final_output.select('customer_id'), ['customer_id'], how='leftanti')
unmatched_sf2 = unmatched_sf.join(final_output.select('sfdc_id__c'), ['sfdc_id__c'], how='leftanti')
unmatched_zd2 = unmatched_zd.join(final_output.select('zd_user_id'), ['zd_user_id'], how='leftanti')

fuzzy_match_shop_sf = test_for_different_combination(unmatched_shop2, unmatched_sf2)
fuzzy_match_shop_zd = test_for_different_combination(unmatched_shop2, unmatched_zd2)
fuzzy_match_sf_zd = test_for_different_combination(unmatched_sf2, unmatched_zd2)


In [0]:
# MATERIALIZE fuzzy match results to temp tables to avoid recomputing
# Write to temp tables 
fuzzy_match_shop_sf.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('silver.customer_master_data_management.temp_fuzzy_match_shop_sf')
fuzzy_match_shop_zd.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('silver.customer_master_data_management.temp_fuzzy_match_shop_zd')
fuzzy_match_sf_zd.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('silver.customer_master_data_management.temp_fuzzy_match_sf_zd')

# Read back from temp tables
fuzzy_match_shop_sf = spark.table('silver.customer_master_data_management.temp_fuzzy_match_shop_sf')
fuzzy_match_shop_zd = spark.table('silver.customer_master_data_management.temp_fuzzy_match_shop_zd')
fuzzy_match_sf_zd = spark.table('silver.customer_master_data_management.temp_fuzzy_match_sf_zd')

print("✓ Fuzzy match results materialized to temp tables")
# display(fuzzy_match_shop_sf.groupBy('customer_id').count())

In [0]:
end_time = time.perf_counter()
duration = end_time - start_time
print(f"Execution time: {duration:.2f} seconds")

In [0]:
# Once i am done looking at the table maybe only do the join on the id after doing the fuzzy matching

# mask = fuzzy_match_shop_sf.filter(F.col('total_confidence') >= 0.85).select('customer_id').distinct()
# display(fuzzy_match_shop_sf.filter(F.col('total_confidence') >= 0.85).groupBy('customer_id').count())

fuzzy_match_shop_sf2 = fuzzy_match_shop_sf.filter(F.col('total_confidence') >= 0.85).drop('soundex_score', 'leven_name_score', 'leven_last_name_score', 'phone_score', 'leven_city_score', 'state_score', 'total_confidence', 'rank')
display(fuzzy_match_shop_sf2)
fuzzy_match_shop_sf2 = fuzzy_match_shop_sf2.select('customer_id','sfdc_id__c')

fuzzy_match_shop_zd2 = fuzzy_match_shop_zd.filter(F.col('total_confidence') >= 0.85).drop('soundex_score', 'leven_name_score', 'leven_last_name_score', 'phone_score', 'leven_city_score', 'state_score', 'total_confidence', 'rank')
display(fuzzy_match_shop_zd2)
fuzzy_match_shop_zd2 = fuzzy_match_shop_zd2.select('customer_id','zd_user_id')

fuzzy_match_sf_zd2 = fuzzy_match_sf_zd.filter(F.col('total_confidence') >= 0.85).drop('soundex_score', 'leven_name_score', 'leven_last_name_score', 'phone_score', 'leven_city_score', 'state_score', 'total_confidence', 'rank')
display(fuzzy_match_sf_zd2)
fuzzy_match_sf_zd2 = fuzzy_match_sf_zd2.select('sfdc_id__c','zd_user_id')


In [0]:
#Join on the ids
combine_fuzzy_matches = fuzzy_match_shop_sf2.join(fuzzy_match_shop_zd2, fuzzy_match_shop_sf2.customer_id == fuzzy_match_shop_zd2.customer_id, how='inner').drop(fuzzy_match_shop_zd2.customer_id)

# #Join on the ids
combine_fuzzy_matches2 = fuzzy_match_shop_zd2.filter(~F.col('customer_id').isin(combine_fuzzy_matches.select('customer_id'))).join(fuzzy_match_sf_zd2, fuzzy_match_shop_zd2.zd_user_id == fuzzy_match_sf_zd2.zd_user_id, how='inner').drop(fuzzy_match_sf_zd2.zd_user_id)

# Combine the 2
final_combine_fuzzy_matches = combine_fuzzy_matches.unionByName(combine_fuzzy_matches2)
display(final_combine_fuzzy_matches)

display(final_combine_fuzzy_matches.groupBy('customer_id').count())
display(final_combine_fuzzy_matches.groupBy('sfdc_id__c').count())
display(final_combine_fuzzy_matches.groupBy('zd_user_id').count())

In [0]:
fuzzy_match_shop_sf2_filtered = fuzzy_match_shop_sf2.filter(~F.col('customer_id').isin(final_combine_fuzzy_matches.select('customer_id'))).withColumn('zd_user_id', F.lit(None))
# display(fuzzy_match_shop_sf2_filtered)

final_combine_fuzzy_matches2 = final_combine_fuzzy_matches.unionByName(fuzzy_match_shop_sf2_filtered)
# display(final_combine_fuzzy_matches2)

fuzzy_match_shop_zd2_filtered = fuzzy_match_shop_zd2.filter(~F.col('customer_id').isin(final_combine_fuzzy_matches2.select('customer_id')))
fuzzy_match_shop_zd2_filtered = fuzzy_match_shop_zd2_filtered.filter(~F.col('zd_user_id').isin(final_combine_fuzzy_matches2.select('zd_user_id')))
# display(fuzzy_match_shop_zd2_filtered)

final_combine_fuzzy_matches3 = final_combine_fuzzy_matches2.unionByName(fuzzy_match_shop_zd2_filtered.withColumn('sfdc_id__c', F.lit(None)))

fuzzy_match_sf_zd2_filtered = fuzzy_match_sf_zd2.filter(~F.col('sfdc_id__c').isin(final_combine_fuzzy_matches3.select('sfdc_id__c')))
fuzzy_match_sf_zd2_filtered = fuzzy_match_sf_zd2_filtered.filter(~F.col('zd_user_id').isin(final_combine_fuzzy_matches3.select('zd_user_id')))


final_combine_fuzzy_matches4 = final_combine_fuzzy_matches3.unionByName(fuzzy_match_sf_zd2_filtered.withColumn('customer_id', F.lit(None)))

display(final_combine_fuzzy_matches4)


# fuzzy_match_shop_zd2_filtered = fuzzy_match_shop_zd2.filter(~F.col('customer_id').isin(final_combine_fuzzy_matches.select('customer_id')))
# fuzzy_match_sf_zd2_filtered = fuzzy_match_sf_zd2.filter(~F.col('zd_user_id').isin(final_combine_fuzzy_matches.select('zd_user_id')))
# #Join on the ids
# combine_fuzzy_matches3 = fuzzy_match_shop_sf2_filtered.join(fuzzy_match_sf_zd2_filtered,